# Symptom Correlation Training
Train a model to predict co-occurring symptoms (multi-label output) and convert to TFLite.

In [ ]:
import numpy as np
from pathlib import Path
from tensorflow import keras
from sklearn.metrics import precision_score, recall_score

ROOT = Path('..')
DATA_DIR = ROOT / 'data' / 'processed' / 'correlation'
X = np.load(DATA_DIR / 'symptom_matrix.npy')
# X is samples x num_symptoms (multi-hot)
num_symptoms = X.shape[1]
# Use X as both input and target (autoencoder-like multi-label prediction)
X_train = X
y_train = X

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(num_symptoms,)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(num_symptoms, activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy')
model.summary()
model.fit(X_train, y_train, epochs=50, batch_size=64, validation_split=0.1)

In [ ]:
# Evaluate precision@5, recall@10 (approximation using thresholds)
pred = model.predict(X_train)
# simple top-k evaluation
def precision_at_k(y_true, y_pred, k):
    correct = 0
    total = 0
    for yt, yp in zip(y_true, y_pred):
        topk = yp.argsort()[-k:][::-1]
        correct += int(sum(yt[topk]))
        total += k
    return correct / total if total else 0
print('Precision@5', precision_at_k(y_train, pred, 5))
print('Recall@10', precision_at_k(y_train, pred, 10))

In [ ]:
# Convert to TFLite
import tensorflow as tf
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
(Path('..') / 'models').mkdir(parents=True, exist_ok=True)
with open(Path('..') / 'models' / 'symptom_correlation_v1.tflite', 'wb') as f:
    f.write(tflite_model)
print('Saved models/symptom_correlation_v1.tflite')